# DSC 102: Systems for Scalable Analytics

## Lecture 9 Slides
### Topic 2: Parallel and Scalable Data Processing
### Part 2: Scalable Data Access

### Memory Hierarchy: I/O Motivation
The numbers that matter: Every data science pipeline lives somewhere in
this pyramid
| Tier | Capacity | Bandwidth | Latency | ~Cost |
| --- | --- | --- | --- | --- |
| CPU Cache (L1–L3) | MBs | ~100 GB/s | 100s of cycles | ~$2/MB |
| DRAM (Main Memory) | ~32 GB | ~30 GB/s | ~100ns | ~$5/GB |
| Flash SSD | ~2 TB | ~3 GB/s | ~100 μs | ~$80/TB |
| Magnetic HDD | ~10 TB | ~200 MB/s | ~10 ms (seek!) | ~$20/TB |


Key Insight:
HDD bandwidth is ~150× slower than DRAM, but ~500× cheaper per TB.
Every DS system must decide where to live in this tradeoff.

### What Happens When Data>DRAM?

The central scalability problem in data science

The Problem

• pandas.read_csv() loads entire file into DRAM — crashes or thrashes
with OOM errors on >DRAM files

• OS virtual memory uses disk as
DRAM 'overflow' — every swap
event is 100× slower

• NumPy array of 100M float64 = 800
MB. A 10 GB CSV will never fit for
most laptops

• Naive code silently degrades:
appears to run, actually hammering
swap


The Solution: Paged Access

• Divide file into fixed-size pages (e.g. 4
KB). Load pages one batch at a time

• Keep only a DRAM 'buffer pool' of active
pages; evict cold pages as needed

• Process data in streaming/chunked
passes instead of all at once

• Enables datasets 10–100× larger than
DRAM with predictable, bounded
memory use


Core idea:
Split the file virtually or physically, then stage reads of pages from disk to DRAM

### Page Access Step-By-Step Mechanics

OS buffer pool sits between disk and DRAM

DISK: File split into pages (P1-P8) --> DRAM — OS Buffer Pool (e.g. 4 frames) (P1-P4)

Filescan example — 4-frame buffer, 6-page file:
1. Read P1→P4
(fill buffer)
2. Read P5:
buffer full → evict P1
(LRU) → cache P5
3. Read P6:
evict P2 → cache P6
4. Total I/O cost:
6 page reads
(each page read once
→ optimal)

↑ Each page read exactly once — best case for sequential (filescan) access

### Quantifying I/O: The Cost Model

I/O Cost =
#pages read + #pages written

Latency =
(I/O Cost × Page Size) / Throughput

Local HDD Filescan

- File: 40 GB | Page: 4 KB |
HDD: 200 MB/s

- Pages = 40 GB / 4 KB =
= 10M pages

- Time = 40 GB / 200 MB/s =
= 200 s

- 15× slower than SSD!


Local SSD Filescan

- File: 40 GB | Page: 4 KB |
SSD: 3 GB/s

- Pages = 40 GB / 4 KB =
= 10M pages

- Time = 40 GB / 3 GB/s ≈
≈ 13.3 s

- ✓ Reference


S3 Remote Read

- File: 40 GB | Page: 4 KB |
Net: 1 GB/s

- Pages = 40 GB / 4 KB =
= 10M pages

- Time = 40 GB / 1 GB/s =
= 40 s

- 3× slower than SSD


Remember critical detail about HDD:
Random I/O (seek + rotational latency) can be 100–1000× slower than sequential.
A 10M-page random read = hours; 10M-page sequential = 200s

### Storage Layouts

Row-Store · Column-Store · Tiled / Blocked

The choice of layout can reduce I/O costs by an order of magnitude
— or blow them up by the same factor.

### A Running Example

In this section, we will be using a running example of a 6x4 DataFrame.

Assume page capacity = 4 cells.
We will trace the same 3 queries
across all layouts

|  | A | B | C | D |
| --- | --- | --- | --- | --- |
| 1 | 1a | 1b | 1c | 1d |
| 2 | 2a | 2b | 2c | 2d |
| 3 | 3a | 3b | 3c | 3d |
| 4 | 4a | 4b | 4c | 4d |
| 5 | 5a | 5b | 5c | 5d |
| 6 | 6a | 6b | 6c | 6d |

Three Benchmark Queries
1. SELECT * FROM df

Full table scan (all cols, all rows)
— loads every cell
2. SELECT SUM(B) FROM df

Column projection
— only column B needed
3. SELECT * WHERE row IN {2,4}

Row lookup
— 2 specific rows, all columns

Page size = 4 cells throughout

### Row-Store (N-Store) Layout

Rows serialized sequentially — one record fully before the next.
Used by: CSV, pandas, Dask DataFrames, OLTP databases.

On Disk: →

P1: 1a **1b** 1c 1d

Row 1

P2: 2a **2b** 2c 2d

Row 2

P3: 3a **3b** 3c 3d

Row 3

P4: 4a **4b** 4c 4d

Row 4

P5: 5a **5b** 5c 5d

Row 5

P6: 6a **6b** 6c 6d

Row 6

Q1:
- Full scan Pages needed: All 6
- I/O cost: 6 
- OK
- Each page loaded once.
- Ideal for full-table analytics.

Q2:
- SUM(B) Pages needed: All 6
- I/O cost: 6
- BAD
- Must read all 24 cells just to get 6 'b' values.
- 4× wasted I/O vs. col-store.

Q3:
- Rows 2,4
- Pages needed: P2, P4
- I/O cost: 2
- OK
- Row lookup is direct — one page per row. Efficient!

### Column Store (DSM) Layout
Each column stored contiguously.
Used by: Parquet, ORC, Arrow IPC, Vertica, Redshift, BigQuery, DuckDB.

Col A: P1 (1a, 2a, 3a, 4a) P2 (5a, 6a, -, -)

Col B: P3 (1b, 2b, 3b, 4b) P4 (5b, 6b, -, -)

Col C: P5 (1c, 2c, 3c, 4c) P6 (5c, 6c, -, -)

Col D: P3 (1d, 2d, 3d, 4d) P4 (5d, 6d, -, -)

Q1: Full scan 
- Pages needed: All 8
- I/O cost: 8
- WORSE
- 8 pages vs 6 for row-
store! More overhead
for a full scan.


Q2: SUM(B) 
- Pages needed: P3, P4 only
- I/O cost: 2
- GREAT
- Only 2 pages!
- Skip A/C/D entirely.
- 3× better than row-store.

Q3: Rows 2,4
- Pages needed: P1,P3,P5,P7 (all cols)
- I/O cost: 4
- OK
- Need 4 pages (one per
column) — worse than
row-store's 2.

### Tiled-Block (PAX) Layout

Split data into 2D tiles.
Used by: Modin, SAP HANA, ScaLAPACK, deep learning tensor frameworks.

P1 rows 1-2, cols  A-B
| | |
| -- | -- |
| 1a | 1b |
| 2a | 2b |

P2 rows 1-2, cols  C-D
| | |
| -- | -- |
| 1c | 1d |
| 2c | 2d |

P3 rows 3-4, cols  A-B
| | |
| -- | -- |
| 3a | 3b |
| 4a | 4b |

P4 rows 3-4, cols  C-D
| | |
| -- | -- |
| 3c | 3d |
| 4c | 4d |

P5 rows 5-6, cols  A-B
| | |
| -- | -- |
| 1a | 1b |
| 2a | 2b |

P6 rows 5-6, cols  C-D
| | |
| -- | -- |
| 5c | 5d |
| 6c | 6d |

I/O Summary
| Query | I/O |
| -- | -- |
| Q1 (full) | 6 |
| Q2 Sum (B) | 3 |
| Q3 rows 2,4 | 4 |

Middle ground —
avoids worst case
for both column-only
and row-only
access patterns.

### Layout Comparison: Head-To-Head

I/O cost comparison — 6×4 table, page size = 4 cells

| Query/Workload | Row-Store | Col-Store | Tiled (2x2) | Winner |
| -- | -- | -- | -- | -- |
| Q1: Full scan (SELECT *) | 6 ✓ | 8 ✗ | 6 ✓ | Row / Tiled |
| Q2: SUM(B) — 1 col | 6 ✗ | 2 ✓ | 3 | Col-Store |
| Q3: Row lookup (2 rows) | 2 ✓ | 4 ✗ | 2 ✓ | Row / Tiled |
| ML training (sample rows) | ✓ natural  | extra join | ✓ natural  | Row / Tiled |
| Analytics — many agg stats | wasteful | ✓ column skip | partial | Col-Store |
| Matrix multiply A×B | poor locality | poor locality | ✓✓ cache-friendly  | Tiled |
| Mixed OLAP + OLTP (*) | ✗ | ✗ | varies | Depends |

Key Principle:

No universally best layout. Match layout to your dominant access pattern.
Parquet (columnar) right for analytics; row-stores for OLTP and ML data loaders.

### Real-World File Formats By Layout

Row-Store

- Formats:
CSV / TSV, 
JSON Lines, 
Avro, 
HDF5 row-major, 
Feather (row mode)

- ✓ Pros:
Fast full-row retrieval, 
 Natural for ML batching, 
 Simple to append new records

- ✗ Cons:
Wastes I/O on column-only
queries, 
Poor compression (adjacent
cells differ)

Col-Store:

- Format: Apache Parquet, 
Apache ORC, 
Arrow IPC, 
DuckDB native, 
Redshift/BigQuery internal

- ✓ Pros:
 Skip columns not needed, 
Excellent compression (same-
type runs), Vectorized execution friendly
- ✗ Cons:
Slow row reconstruction (need
all cols), Append is expensive (rewrite
column chunks)

Tiled:

- Formats: HDF5 chunked datasets, 
NetCDF-4 (chunked), 
Zarr, 
TileDB, 
NumPy memmap (tiled)
- ✓ Pros:
Cache-friendly for 2D access, Good for matrix/tensor ops, Balanced row/column queries
- ✗ Cons:
 Tile size must be tuned to
workload, 
More complex metadata, Less common tooling support

Practical Advice: Use Parquet for analytics pipelines; CSV/Avro for data ingestion; chunked HDF5/Zarr
for scientific/array data.

### Cache Replacement Policies

### (Page) Cache Replacement Policies

In a little more detail...

LRU (Least Recently Used)
- ✓ Good for: repeated-access patterns (joins, nested loops)
- ✗ Bad for: filescan — repeatedly evicts what we'll never see again

MRU (Most Recently Used)
- ✓ Good for: filescan on large files — keep the 'older' context
- ✗ Bad for: workloads with locality (temporal reuse)

Clock (Approx LRU)
- ✓ Good for: OS implementations — efficient approx of LRU w circular buffer. Used in Linux
- ✗ Lower accuracy than true LRU

ML-based Policies
- ✓ Good for: complex mixed workloads; can learn access patterns
- ✗ Overhead in training/inference; not yet mainstream in DBs

Insight: The 'best' policy depends on access pattern — LRU is suboptimal for filescan!
MRU wins there.

### LRU: An Example & Counter-Example

Case 1: Repeated access pattern:

Access sequence: P1,P2,P3,P4,P1,P2,P3,P4,P5
LRU evicts P1 when P5 accessed — P1–P4 hot
pages stay cached

Total misses: 5 (optimal)

Case 2: Filescan (LRU's worst case):

Access: P1,P2,P3,P4,P5,P1,P2,P3,P4,P5...

4-frame LRU: every access is a miss! LRU
evicts the page about to be requested next.

Total misses: 10 (worst case!)

MRU would achieve 5 misses (optimal) on
this pattern
because it keeps the 'seen less recently'
pages that will appear next in the cycle.

Note the Dask implication:
Dask processes partitions sequentially — each
partition is like a filescan page. OS LRU on
partition files can hurt; using bounded memory
with explicit partition sizes is critical.

### Dask Architecture Deep Dive
How Dask DataFrames scale pandas operations beyond DRAM

Task graphs · Partitions · Schedulers · Lazy evaluation

### Dask Architecture overview

Three-layer stack: user API → task graph → scheduler + workers

User API Layer
- dask.dataframe, dask.array, dask.bag, dask.delayed
- Mirrors pandas / NumPy API — minimal code changes needed

Task Graph Layer
- Lazy evaluation: operations build a
Directed Acyclic Graph (DAG)
- No work until .compute() called.
Optimizer can fuse/prune tasks.

Scheduler +
Workers
- Synchronous, threaded,
multiprocessing, or distributed
scheduler
- Workers hold partitions in DRAM;
spill to disk when DRAM full

Data Flow:
file.parquet
(disk) --> 
Dask DF
(row-store split) --> 
Task DAG
(lazy ops) --> 
Partitions in
DRAM (workers) -->
Result
.compute()

### Dask DataFrame
Dask DataFrame partitions can be considered as row-store splits.
Each partition is a full Pandas DataFrame held in DRAM or spilled to disk.

Key Properties
- Partitions ordered by
index by default
- Each partition
independent (can be
parallelized)
- Operations applied per-
partition, then aggregated
- Lazy: partition not loaded
until .compute()
- Spill to disk if workers run
out of DRAM

Dask uses row-store layout — all columns are loaded per partition even for column-only
queries. Use Parquet with column pruning to mitigate

### Dask Task Graphs

Through the Dask task graphs we can see lazy evaluation in action

Benefits of Lazy Evaluation
- Fusion: filter + groupby can be done in
one pass over a partition
- Pruning: unused partitions never loaded
- Reuse: shared subgraphs computed once

In [ ]:
import dask.dataframe as dd
df = dd.read_csv('data/*.csv')
df2 = df[df['value'] > 0]
result = df2.groupby('key').sum()
out = result.compute() # ← triggers all

### Dask Scheduler: Choose Right Engine

Scheduler executes the task graph. Wrong choice = performance cliff

Synchronous
- dask.config.set(scheduler='synchronous')
- When: Debugging. Single-threaded, step-through-able.
- ✓ Easy to debug; no overhead | ✗ No parallelism whatsoever

Threaded
(default)
- dask.config.set(scheduler='threads')
- When: Pure Python operations (pandas ops, string ops).
- ✓ Low overhead; shared memory | ✗ GIL limits true parallelism for CPU-bound ops

Multiprocessing
- dask.config.set(scheduler='processes')
- When: CPU-bound operations on numeric data.
- ✓ True CPU parallelism; bypasses GIL | ✗ Serialization overhead; memory not shared

Distributed
 (dask.distributed)
- from dask.distributed import Client; c = Client()
- When: Multi-node clusters or local cluster with monitoring.
- ✓ Dashboard; adaptive scaling; spill-to-disk | ✗ Network overhead; setup complexity

For data science on a single node: start with 'threads'; switch to 'processes' for heavy
NumPy/Numba; use distributed for the dashboard and memory management.

### Dask Memory Mgmt And Disk Spills
What happens when partitions exceed available DRAM

Dask Distributed Memory Lifecycle
- 0–60% Normal operation
- 60–70% Spill to disk begins
- 70–80% Pause workers
- 80–95% Terminate tasks

Config:
distributed.worker.memory.{target,spill,pause,
terminate}

Spilling: What Goes Wrong
- Spilled data written to local disk — not
remote storage. Disk I/O during
computation degrades throughput badly.
- Workers spill by partition, not by page —
a 2 GB partition either fits in DRAM or
doesn't. No partial eviction.
- Spill location defaults to /tmp — make
sure it's on a fast SSD, not a network
mount.
- Spilling degrades to ~200 MB/s HDD vs.
~10 GB/s DRAM; this is a 50× slowdown.
Avoid at all costs.

Best practice: size partitions so that N_workers × partition_size < 60% of total DRAM.
Never let Dask spill in production.

### Dask Tuning Guide
Practical knobs to make Dask fast in practice

Partition sizing · File formats · Column pruning · Shuffle · Persist · Profiling

### 1. Partition Size: Goldilocks Problem

Too few = underutilized cores. Too many = scheduler overhead dominates.

Too small
(<10 MB), Sweet spot: 100 MB – 1 GB per partition, Too large
(>2 GB)

Too Few / Too Large
• Worker runs out of DRAM — triggers spilling
(50× slowdown)
• Can't parallelize: 4 cores but only 2 partitions
= 2 idle cores
• Operations like groupby, join must shuffle
huge partitions
Too Many / Too Small
• Scheduler overhead dominates: 10,000 tasks
of 1MB each = slow
• Metadata overhead: Dask tracks each
partition's index range
• Function call overhead per partition becomes
bottleneck

Rule of thumb: aim for 100–500 MB partitions. After filters or merges, always repartition() to
re-balance.

In [ ]:
# Check current partition sizes
df.map_partitions(lambda p: p.memory_usage(
deep=True).sum()).compute()
# Repartition to ~100MB chunks
df2 = df.repartition(partition_size='100MB')
# Repartition to specific count
df3 = df.repartition(npartitions=50)
# After filter, consolidate small parts
df_filtered = df[df.value > 0]
df_filtered = df_filtered.repartition(
partition_size='200MB')

### 2. File Formats: Use Parquet, Not CSV
Format choice can change I/O cost by 5 – 20x for analytical workloads

| Format  | I/O vs CSV  |  Compress | Strong Types  | Predicate Push  | Column Prune  |
|---|---|---|---|---|---|
| CSV  | 100%  | None  | no  |  no | no  |
|  JSON Lines |  120% | none  | partial  |  no | mo  |
| Parquet  | 15–30%  |  Snappy/Zstd | yes  | yes  | yes  |
|  ORC | 12–25%  | Zlib/Snappy  | yes  | yes  |  yes |
|  Feather/Arrow |  80%  | LZ4  | yes  | no  | yes  |

In [ ]:
# Write and read with column pruning + predicate pushdown
df.to_parquet('data/', engine='pyarrow', compression='snappy', partition_on=['year'])
dd.read_parquet('data/', columns=['A','B'], filters=[('year', '==', 2023)])

### 3. Persist, Shuffle, Anti-Patterns

Three more levers between fast and catastrophically slow:
persist() — Pin hot data in DRAM

- Problem: df.compute() re-reads + re-computes from disk every call
- Solution: df_hot = df.persist() — materializes and keeps in DRAM
- Use for: DataFrames accessed multiple times (e.g. train/val split reuse)
- Warning: only works with dask.distributed. Fills worker DRAM — check sizes first.

shuffle — The expensive bottleneck
- groupby, merge/join, set_index all require data redistribution across partitions
- Default shuffle='tasks': generates O(P²) tasks — very slow for P > 100 partitions
- Better: shuffle='p2p' (Dask distributed ≥2022.8) — peer-to-peer, much faster
- Even better: set_index on the join key before joining — eliminates shuffle

Anti-patterns to avoid
- df.apply(fn, axis=1) : Runs fn in Python per row — disables vectorization
- for partition in df.partitions: partition.compute() — defeats lazy eval
- df.compute() early: materializes whole df before needed computation
- Mixing Dask and pandas mid-pipeline without explicit boundaries

### Best-Practice Pattern: Dask+Parquet
Combining all tuning advice into a complete, scalable DS pipeline

- Step 1: Column pruning → skip I/O for unused columns.
Predicate pushdown → skip row groups on disk.
- Step 2: Repartition after predicate filter — data was
pruned; partitions may now be unbalanced.

In [ ]:
# ── End-to-end best-practice template ──────────────────────────
from dask.distributed import Client
import dask.dataframe as dd
client = Client(n_workers=4, threads_per_worker=2,
memory_limit='4GB') # 4 workers × 4GB = 16 GB DRAM
# 1. Read Parquet with column pruning + predicate pushdown
df = dd.read_parquet('s3://bucket/data/*.parquet',
columns=['user_id','revenue','ts'],
filters=[('ts', '>=', '2024-01-01')])
# 2. Repartition to ~200 MB chunks (avoids spill)
df = df.repartition(partition_size='200MB')
# 3. Heavy op — set_index first to avoid later shuffle
df = df.set_index('user_id', sort=True)
# 4. Persist if reused; compute once at end
result = df.groupby('user_id')['revenue'].sum().compute()

### Summary & Key Principles

1. I/O cost dominates for large datasets. Reducing page reads is the #1 optimization lever.
2. Layout determines I/O cost.
Row-store for row access; Column-store for analytics; Tiled for 2D.
3. No universal best layout. Match layout to the dominant access pattern of your workload.
4. Cache replacement matters:
LRU is not always optimal — filescan workloads need MRU-like policies.
5. Dask = Pandas API + row-store partitions + lazy DAG.
The partition is the unit of parallelism and spilling.
6. Three Dask levers: (a) right partition size 100–500 MB
(b) Parquet + column pruning
(c) avoid shuffle

## Lecture 10 Slides: Scaling Data Science Operations
Applying I/O cost principles to real data science programs

### DB Operations on Disk
Project · Aggregate · GROUP BY · Select

The Recurring Pattern: filescan + DRAM accumulator + I/O cost = read pages + write pages



### Non-deduplicating Project – SELECT C FROM R

SELECT C FROM R (return all values of column C; keep duplicates)

Row-store (NSM)
- ✗ All 6 pages read — A, B, D are
wasted I/O!
- I/O cost: 6 (read) + output pages
(write)

Col-store (DSM)
- ✓ Only 2 pages read — skip A, B, D entirely!
- I/O cost: 2 (read) + output pages (write) ← 3× fewer reads!

This is why Parquet (col-store) exists — analytics queries touch only a few columns; row-store wastes 75% of I/O here

### Simple Aggregates – SELECT MAX(A) or SUM(D) FROM R

Key Concept: Running Accumulator

Simple aggregates (MAX, MIN, SUM, COUNT, AVG) require only a single scalar "running value" in DRAM. No maNer how large the dataset,
the DRAM footprint is O(1) — one number. The filescan reads each page once, updates the running value, and discards the page.

Row-store — SELECT MAX(A) FROM R
- Read P1: 1a,1b,1c,1d → max=1a
- Read P2: 2a,2b,2c,2d → max=2a
- Read P3: 3a,3b,3c,3d → max=3a
- …
- Read P6: 6a,6b,6c,6d → max=6a

I/O: 6 reads (all pages) + 1 write (result)

⚠
reads cols B,C,D
unnecessarily

Col-store — SELECT MAX(A) FROM R
- A-P1: 1a,2a,3a,4a
- A-P2: 5a,6a,—,—
- B, C, D pages — skipped entirely!

Running MAX: null → 1a → 2a → 3a → 4a →
5a → 6a ✓

I/O: 2 reads (A pages only) + 1 write 3× fewer reads

### GROUP BY Aggregate – SELECT A, SUM(D) FROM R GROUP BY A

SELECT A, SUM(D) FROM R GROUP BY A ← not a simple scalar; need one running sum per group

- One hash table entry per unique group
— DRAM footprint = O(# distinct
groups)
- 🔁 Single filescan: read each page once,
update running sum for the page's
group
- ✓ I/O cost: 6 pages read + output pages
written (hash table result) — just one
filescan!

⚠ What if hash table > DRAM? Program likely crashes or thrashes — see next slide!

### GROUP BY Overflow – When the Hash Table Exceeds DRAM

The Problem: Thrashing

If # distinct group values is huge (e.g. GROUP BY user_id with 100M users), the hash table won't fit in DRAM. The OS
keeps swapping hash table pages to/from disk — this is called thrashing. Performance collapses from O(N) reads to
O(N²) or worse.

Solution: Divide & Conquer — Split R by group values

- Trick 1: Hash-partion R into buckets by group key — each bucket's HT fits in DRAM alone.
- Trick 2: Reduce running info early (store only sum/count, not all tuples).

### Relational Select – σ B="3b" (R) — Filter Rows by Predicate

SELECT * FROM R WHERE B = "3b" → return only rows where column B equals "3b"

LRU eviction: When cache is full, evict the
least recently used page (P1, then P2, …).

Good choice here: filescan accesses each page exactly
once, so oldest page is safest to evict.

I/O cost: 6 reads + output pages writes

### Matrix & Tensor Algebra
Sum/Norms · Scalable Tiled Layout · Gramian M<sup>T</sup>M

Key insight: matrix operations map naturally onto filescan with running accumulator — and 2led layout minimizes I/O for
2D access patterns

### Scalable Matrix Sum & Frobenius Norm – ||M||²_F
|M||²_F = Σij m² ij (sum of squares of all entries) — also called Frobenius norm square

DRAM footprint: O(1)
- Accumulator is a single scalar — one
integer. Works even if M is 100 TB
on disk.

Access pattern
- Pure filescan — read each page
once, square values, add to running
sum, discard page.

I/O cost
- 6 reads (all pages) + 1 write
(scalar result). Same
formula as SUM aggregate.

Tiled layout advantage
- For block matrix ops (mulEply,
Gram), Eled layout reduces I/O —
see next slides.

### Scalable Matrix Algebra – Why Tiled Layout Wins

For matrix algebra (multiply, transpose, Gram), a row of the output depends on BOTH rows and columns of the
input — row-store means reading entire matrix multiple times. Tiled layout aligns on-disk access to the 2D access
pattern.

Row-store
- Computing C[i,j] = Σ k A[i,k]·B[k,j] needs row i of A and column j of B.
Column j of B is scattered across all pages in row-store → many page reads!
- I/O: For 6×4 matrix: up to 6 page reads per output cell
- ❌ Poor fit for matrix mulGply & Gram

Col-store
- Column j of B is conEguous — fast to fetch. But row i of A is now scaWered.
Transpose trick helps: A T is col-store of A.
- I/O: Better for one dimension, but still asymmetric
- ⚠ Partial fit — good for Gram (AT·A)

Tiled (2×2)
- Each tile covers a 2×2 block. Computing one output tile only reads the tiles that overlap —
nearby data is co-located on disk.
- I/O: Gram (MT M) on 6×4 tiled: 18 reads + 4 writes
Scales on BOTH matrix dimensions!
- ✓ Best general fit for matrix algebra

### Gramian Matrix M<sup>T</sup>M – Tiled Layout Walkthrough

M<sup>T</sup>M (Gram matrix) appears in: linear regression normal equa:ons, PCA covariance, SVD — Output is 4×4 (for 6×4 M)

Key: never need > 2 tiles in DRAM at once → constant DRAM footprint

Max I/O cost: 3+3+6+6 = 18 reads + 4 writes | Scalable on BOTH matrix
dimensions!

O1 = ATA + CTC + ETE
- Read tiles A, C, E (one by one) → compute incrementally → write O1
- 3 reads + 1 write

O4 = B^TB + D^TD + F^TF
- Read tiles B, D, F → compute incrementally → write O4 
- 3 reads + 1 write

O2 = A^TB + C^TD + E^TF
- Read (A,B) → parEal O2; read (C,D) → add; read (E,F) → add; write O2
- 6 reads + 1 write

O3 = B^TA + D^TC + F^TE
- Read (B,A), (D,C), (F,E) pairs → accumulate → write O3 
- 6 reads + 1 write

### Gradient Descent in ML
ERM · BGD · SGD · Access Patterns

Gradient descent iteratively minimizes a loss function — each step is a filescan over training data

### Numerical Optimization in ML – The Setup
Empirical Risk Minimization (ERM)
- Most ML classificaJon/regression models minimize a loss over training examples:
w* = argmin_w Σ_i l(yᵢ, f(w, xᵢ)) where l() measures predicJon error for one example (yᵢ, xᵢ)

Model weights
- The parameters we are optimizing (a
vector). Small — fits in DRAM. e.g. 1
weight per feature.

Feature vector
- Input features for training example i.
Could be high-dimensional (images,
text embeddings).

Label
- True output for example i. Scalar for
regression; class id for classificaEon.

Loss function
- Measures how wrong the prediction is.
Common: squared error (regression),
log-loss (classification).

Prediction fn
- For Generalized Linear Models: f(w,x) =
wᵀx (dot product). Very cheap — O(d)
ops per example.

Total loss
- Sum of l() over all n examples.
Minimizing L(w) → find w*.

Why iterative? Closed-form solution (e.g. matrix inversion) costs O(d³) — infeasible for d = millions of features. Gradient descent is O(nd) per step

### Batch Gradient Descent (BGD) – Algorithm & Update Rule

0. Initialize

    Set w⁽⁰⁾ to some starting value (e.g. zeros or random small values)

1. Compute Gradient

    ∇L(w⁽ᵏ⁾) = Σᵢ ∇l(yᵢ, f(w⁽ᵏ⁾, xᵢ)) — sum over ALL n training examples

2. Update Weights

    w⁽ᵏ⁺¹⁾ ← w⁽ᵏ⁾ − η · ∇L(w⁽ᵏ⁾) where η is the learning rate

3. Check Convergence

    Repeat steps 1–2 unJl ||∇L|| < ε or max epochs reached

Loss Surface & Gradient Steps

η (learning rate) & # epochs: hyperparameters — chosen by user or
AutoML tuning

### BGD Data Access Pattern & I/O Cost – It's Just a Filescan!

Gradient = Σᵢ ∇l(yᵢ, f(w,xᵢ)) — a SUM over per-example vectors, analogous to SQL SUM over
column values. w stays in DRAM; D scanned from disk.

One Epoch of BGD = One Complete Filescan of D

Read P1
(rows 1-2)
-->
compute
∂l/∂w for
rows 1-2

Read P2
(rows 3-4)
-->
accumulate
∇L so far

Read P3
(rows 5-6)
-->
accumulate
∇L final

Update w
w ← w − η·∇L
(in DRAM;
no disk I/O)

Gradient ≈ SQL SUM:
- Each example contributes one term to the gradient sum
— exactly like compuEng SUM(col) in a SQL aggregate
filescan.

I/O cost per epoch: 
- 6 pages read (full filescan of D) + output pages for final
w. No random access — pure sequential.

w is DRAM-resident: 
- Model weights w stay in memory across all page reads.
Only gradients accumulate in DRAM between pages. 

Monitoring convergence: 
- Compute L(w) every epoch — also a filescan. Budget 2×
per-epoch I/O if tracking loss curve.

### Stochastic Gradient Descent (SGD) – Faster Convergence

Two Key Cons of Batch GD
- ✗ Slow convergence: 
BGD often takes thousands of epochs to converge — each epoch = 1
full filescan of D.
- ✗ Costly full-batch gradient: 
Each weight update requires a full filescan: n I/Os per update. With n =
100M rows, every step is expensive.

SGD: Use Mini-batches to Approximate
- ✓ Approximate gradient: 
Use a random sample (mini-batch) of size B ≪ n to esJmate gradient. Noisy
but direcJonally correct.
- ✓ Orders more updates: 
With n=1M and B=100: 10,000 weight updates per epoch vs 1 for BGD.
Converges much faster.
- ✓ Non-convex friendly: 
Mini-batch noise helps escape local minima/saddle points → works well for
deep learning!

SGD Update Rule (per mini-batch):
- ∇L~(W) = Σ_{i∈B} ∇I(yᵢ, f(W, xᵢ)) (sum over mini-batch B, not full dataset)
- w⁽ᵏ⁺¹⁾ ← w⁽ᵏ⁾ − η · ∇L~(w⁽ᵏ⁾) (same form as BGD; just uses approximate gradient)

SGD = "workhorse of modern
ML/DL"

### SGD Access Pattern – Shuffle, Epoch, Mini-batches

Step 1
- Original
Dataset D
- n examples,
on disk

Step 2
- Random
Shuffle
(ORDER BY
RAND())
- ~1–2 passes
over file

Step 3
- Randomized
Dataset D'
- Same data,
new order

Step 4
- Epoch 1:
Seq. scan
into mini-
batches
- n/B updates
per epoch

Step 5
- (Optional)
Re-shuffle
before
next epoch
- or reuse D'

Zoom in: What happens inside one epoch (sequential scan of shuffled D')
r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12
Mini-batch 1
(B=4 examples)
Update w⁽0⁾ Mini-batch 2
(B=4 examples) Update w⁽1⁾ Mini-batch 3
(B=4 examples) Update w⁽2⁾
→ 3 updates this epoch (n=12, B=4) vs BGD: 1 update per epoch

### SGD I/O Cost & BGD vs SGD Tradeoff Analysis
I/O Cost per Epoch of SGD = Shuffle Cost + Filescan Cost (same # bytes as BGD per epoch!)

Shuffle Cost (External Merge Sort)
- Random shuffling of an on-disk file is non-trivial.
- Requires "external merge sort" (out of scope for this course).
- Typically: 1–2 full passes over the file.
- So shuffle cost ≈ 1–2× filescan cost.
- Common opJmizaJon: shuffle once upfront, reuse across all epochs.

Filescan Cost (Mini-batch Compute)
- Sequential scan of shuffled D: 1 filescan per epoch.
- As scan proceeds: count examples, accumulate per-example gradient for
current mini-batch.
- After B examples: update w, reset gradient accumulator.
- Typical B: 32–1024 (powers of 2).

Shuffle-once-upfront vs every-epoch: trade gradient correlation across epochs (slight quality loss) for saving 1 filescan per epoch

| property  | batch gd (bgd)  |  stochastic gd (sgd) |
|---|---|---|
|  Updates per epoch | 1 (one full gradient)  | n/B (one per mini-batch)  |
| Gradient quality  | Exact   | Approximate (noisy)  |
|  I/O per update | n pages (full scan)  | B pages (mini-batch)  |
| I/O per epoch  | n pages  | n pages + shuffle  |
|  Convergence speed | Slow (few updates)  | Fast (many updates)  |
| Non-convex losses  |⚠ Can get stuck| ✓ Noise helps escape  |
| Shuffle cost  | None  | 1–2× filescan (upfront ok)  |

### Worked Examples – Testing Your Understanding
Q1: 100 GB Parquet file, 20 equal-width columns. You compute a sum over 4
columns. What is the I/O cost in GB?
- Hint: Parquet = col-store → only 4 columns read.
- Answer: 100 GB × (4/20) = 20 GB
- Each column = 5 GB. You read 4 columns. Col-store skips the other 16.

Q2: 1 TB matrix in tile format, shape 2000×500 tiles. Compute full matrix sum.
I/O cost in GB?
- Hint: Full matrix sum = read every element once → every tile read once.
- Answer: 1 TB (all tiles, once)
- Matrix sum reads every cell. Tile format doesn't help for a full scan — every tile is
needed. I/O = 1 TB

Q3: 100 million training examples, mini-batch B=50. How many SGD weight
updates in 20 epochs?
- Hint: Updates per epoch = n/B. Total = (n/B) × epochs.
- Answer: (100M / 50) × 20 = 40 million updates
- 2M updates per epoch × 20 epochs = 40M total model updates. BGD would do 20.

Q4: Shuffle-once-upfront vs shuffle-every-epoch for SGD: what is the precise
runtime tradeoff?
- Hint: Shuffling = 1–2 full passes. Epochs = K.
- Answer: Save (K−1) × shuffle cost; trade mini-batch correlaGon
- Shuffle-once saves (K−1) shuffle passes. But epoch 2+ mini-batches are in the
same order → slight gradient correlaEon. Usually acceptable.

### The Universal Pattern – Every Operation Revisited
Operation Disk Access Pattern DRAM Role I/O Cost Key Insight
SELECT C
FROM R
filescan
(row: all pages
col: C pages only)
discard col A,B,D
on the fly
row: N
col: N/cols col-store
SELECT MAX(A)
FROM R
filescan
(row: all pages
col: A pages only)
running MAX
(1 scalar)
row: N
col: N/cols col-store
SELECT A,SUM(D)
GROUP BY A
filescan
(any layout)
hash table
1 row/group
N + output
*worst: thrash* handle overflow!
SELECT *
WHERE B="3b"
filescan +
LRU eviction
output buffer + LRU frames N + output row-store ok
||M||²_F
matrix norm
filescan
(row/tiled)
running sum
(1 scalar) N any layout
M^T M
Gram matrix
tiled pairs
(blocked scan)
2 tiles at once
incremental O
18r+4w
(6×4 ex.) tiled layout
BGD
gradient
filescan/epoch
w in DRAM
model w + running ∇L
N per epoch
× K epochs col-store if sparse
SGD
mini-batch
filescan +
shuffle
model w + mini-batch ∇L
N + shuffle
per epoch more updates/epoch

Across all operations: filescan + small DRAM accumulator = scalable to any dataset size. Layout choice determines which pages to skip.

## Lecture 11 Slides: Distributed Computing for Data Scientists

BSP • MapReduce • Hadoop & HDFS • Spark & RDDs

### Scaleable Data Access – Problem 1
You are given a relation R(X,Y) wherein X is a string datatype of length 22 bytes, while Y is a
float64 datatype of length 8 bytes. Suppose the relation instance has p tuples but only q
unique values of X. What is the minimum
uncompressed size of the output relation (in MB) of the following SQL query?
SELECT X, SUM(Y) FROM R GROUP BY X

1. p = 100 million; q = 20 million
2. p = 1 billion; q = 5 million

### Scaleable Data Access – Problem 2
You are given a database with instances of two relations R(A,B) and
S(B,C,D,E), wherein B is a primary key in S and foreign key in R. The set
of values of B in both relations are identical. All attributes in this database
are of integer datatype (4 bytes each). All tables are stored in column store
format without compression on disk. The number of pages on disk of R and
S are 28 million and 4 million, respectively. What is the rough disk I/O cost
(in pages) of the following query? Exclude output write costs. Assume the
DRAM cache is initially empty.
SELECT MAX(C) FROM S;


### Scaleable Data Access – Problem 3

You are given a relation R(W,X,Y,Z) wherein all 4 attribute data types are of the same
fixed length. Suppose the total size of the file on disk is 12 million pages. What is the
rough I/O cost (in millions of pages) of this SQL query when the file is laid out with the
given layout?

SELECT MIN(Y), MAX(Z) from R

1. Row-major layout
2. Column-major layout

Scaleable Data Access – Problem 4

You are given a large float64 matrix M of dimension 25 million x 10 million stored in
a tiled layout without compression. The tiles are squares of dimensions 2000 x
2000. They stride along both dimensions by 2000 cells starting from top left.
Assume a tile’s data fits exactly on one page on disk. What is the rough minimum
disk I/O cost (in pages) of the following linear algebra computation? Ignore output
write costs. Assume the DRAM cache is initially empty.

Summation of
M[1 : 5,000,000][2,000,001 : 4,000,000] Note that M[i:j][k:l] means only rows i to j
and columns k to l (ends included) are read. Row/column indices start from 1.

### Why we need scalable analytics
Modern datasets outgrow what a single machine can hold or process in reasonable time.

TB → PB
D A T A S E T S I Z E S
- Logs, sensors, images, genomics all push past
single-node memory and disk.

10⁹+
R E C O R D S / D A Y
- Web events, transacnons, IoT streams arrive
faster than one CPU can consume.

Hrs → Min
L A T E N C Y T A R G E T S
- Model retraining, dashboards, and ETL jobs
must finish on schedule.

Single-machine pandas / NumPy hits a wall — we need to spread work across many machines.

### Three flavours of parallelism
We've seen SIMD and task-parallel dataflow. BSP is a third — for distributed clusters.

S I M D
Data parallelism
op: +
one instruc+on → many lanes
Same operaUon, many data elements at once.
Where you've seen it: GPUs, vector units, NumPy
broadcasts.
e.g. add two arrays of 1M floats — every lane does
+, in lockstep

T A S K / D A T A F L O W
Task parallelism
A B
C
D
different tasks, wired by dependencies
Different tasks run in parallel, wired by data
dependencies.
Where you've seen it: Pipelines, Airflow DAGs,
mulnthreaded apps.
e.g. load → clean → featurize → train → eval,
stages overlap

B S P ( n e x t )
Bulk synchronous
compute → barrier → compute
Many workers do the same step, then sync at a
barrier.
Where you've seen it: MapReduce, Spark, Pregel,
distributed ML.
e.g. every worker updates its partition, then
everyone exchanges

All three appear in real data-science work — today we focus on the third.

### Two ways to handle more data
Buy bigger machines or buy more machines.

S C A L E U P ( v e r t i c a l )
Bigger box
- More CPU, RAM, disk in one server
- Simple programming model — still one machine
- Hard ceiling: hardware can only get so big
- Expensive at the top end; single point of failure

S C A L E O U T ( h o r i z o n t a l )
More boxes
- Many commodity machines in a cluster
- Linear-ish scaling: add nodes, get more throughput
- Tolerates node failures by replication
- Needs new programming models — that's our topic

Distributed analytics frameworks all assume horizontal scaling.

### Why distributed computing is hard
Programs that work on one machine break in surprising ways across many.

Network is slow
- Sending data across machines is 1000× slower than local
memory access.

Machines fail
- In a 1000-node cluster, something is always broken.
Software must keep going.

Coordination
- Workers must agree on when to start, stop, and
exchange intermediate results.

Load balance
- A few slow workers (stragglers) hold up the entire job.
Skew is the enemy.

### Bulk Synchronous Parallel (BSP)
Leslie Valiant, 1990 — a bridging model for parallel computation.

The idea
- Run many processors in parallel but make them periodically
stop and synchronize.
- Computation proceeds in supersteps. Inside a superstep,
processors work independently. Between supersteps, they
exchange messages and wait at a barrier.

Why we care
- MapReduce, Pregel, and many Spark patterns are BSP under
the hood. Understanding BSP makes the rest click

### P A R T 1 — B S P: Anatomy of a superstep

Three phases, in order, every time.

1. 
Local computation

Each processor works on its own data. No
coordination, full speed.
→

2. 
Communication

Processors send messages to each other.
Data flows across the network.
→

3. 
Barrier sync

Everyone waits until all messages arrive and
all processors finish.

…then repeat until the algorithm converges or runs out of input.

### Why the barrier matters
Synchronization gives BSP its predictability — and its cost.

W H A T T H E B A R R I E R B U Y S
Deterministic progress
- All messages from superstep N arrive before
superstep N+1 starts.

Easy reasoning
- No race conditions inside a superstep — each
processor sees a consistent snapshot.

Predictable cost model
- Runtime ≈ compute + communicaGon + barrier per
superstep.


W H A T T H E B A R R I E R C O S T S
Stragglers stall everyone
- The whole job moves at the speed of the slowest
worker.

Idle time at the barrier
- Fast workers wait, wasting CPU cycles.

Communication bursts
- All messages travel at the same time — network gets
hammered.

### What if we removed the barrier?
Two workers count word frequencies in a shared counter — same code, different answer every run.

Setup: shared dictionary counts = {"the": 5} • both workers run: counts["the"] = counts["the"] + 1 • expected answer: 7

Without a barrier: computation collapses into dataflow — the answer depends on who finishes first. That non-determinism is a race condition.

### BSP in action: iterative graph algorithms
Think PageRank on a web graph — each node updates from its neighbours.

Setup
- Partition vertices across machines. Each machine holds some
nodes and their outgoing edges.

Each superstep
1. Compute: each vertex calculates its new rank from messages
received last round.
2. Communicate: each vertex sends its updated rank to all out-
neighbours.
3. Barrier: wait for all messages to be delivered.

Termination
- Stop when ranks change by less than ε, or after a fixed number
of supersteps.

### BSP — when it works and when it hurts

Strengths
- Conceptually clean — easy to design and debug
parallel algorithms
- Deterministic — same input gives same output,
no race conditions
- Cost model lets you predict scaling behaviour
- Maps naturally to ML training loops and graph
algorithms
- Many real frameworks are BSP under the hood

Weaknesses
- Barriers waste time on heterogeneous clusters
- One straggler delays the whole job
- Not ideal for low-latency or streaming
workloads
- Communication bursts can saturate the network
- Asynchronous models (e.g. parameter servers)
can be faster for ML

### MapReduce
A pattern that took over data processing in the 2000s.

G o o g l e , 2 0 0 4: 
Engineers at Google were wriGng hundreds of custom distributed programs to index the web. The paqerns kept
repeaGng: split input, process pieces in parallel, combine results.
They abstracted it into two funcGons — map and reduce — and built a framework that handled parallelism, fault
tolerance, and data distribuGon automaGcally. Programmers wrote only the two funcGons.

### Two functions you write
Everything else — distribuMon, parallelism, fault tolerance — is handled by the framework.

map

(key, value) → list of (key, value)
- Takes one input record
- Emits zero or more intermediate (key, value)
pairs
- Runs in parallel on every record — no shared
state
- Stateless: same input, same output, every time

reduce

(key, list of values) → list of (key, value)
- Receives all values that share the same key
- Aggregates them into a smaller result
- Different keys reduced in parallel, by different
workers
- Common ops: sum, count, max, average, join

### MapReduce execution flow
Input
- split into blocks
- raw data on disk

▶

Map
- parallel per block
- user-written

▶

Shuffle
- group by key
- framework

▶

Reduce
- aggregate per key
- user-written

▶

Output
- to HDFS
- results on disk

Key insight: you write only Map and Reduce. The framework handles input splitting,
scheduling, shuffle, sort, and fault recovery.

### The canonical example: word count
Count how often each word appears across a huge collecMon of documents.


In [ ]:
map(doc_id, text):
for word in text.split():
emit(word, 1)
reduce(word, counts):
emit(word, sum(counts))

A s m a l l t r a c e

Input: "the cat sat", "the dog ran"

After map:

(the,1) (cat,1) (sat,1)

(the,1) (dog,1) (ran,1)

After shuffle (grouped by key):

the → [1,1] cat → [1] sat → [1]

dog → [1] ran → [1]

After reduce:

the=2 cat=1 sat=1 dog=1 ran=1

Same code scales from 2 docs to 2
billion docs.

### The shuffle: where the cost lives
The hidden phase between map and reduce — and usually the bottleneck.

What it does
- Takes every map output (k, v) and routes it to the reducer
responsible for key k.

Mechanics
- Partition by hash(key) mod R. Sort each partition. Stream over the
network to the reducer.

Why it dominates runtime
- Touches the disk and network. For large jobs, the shuffle is usually
the bottleneck — not the map or reduce code.

Optimization
- Use a combiner — a local pre-aggregation on map outputs — to
shrink data before the shuffle.

### MapReduce: what it nailed, what it didn't
What it nailed
- Massive scale — clusters of thousands of nodes
- Fault tolerance — re-run failed tasks
automatically
- Simple API — two funcGons to learn
- Locality — schedule tasks where the data lives
- Hides distribuGon complexity from the
programmer

What hurt
- Disk-heavy — writes between every map and reduce
- IteraWve algorithms are painful (must chain MR jobs)
- Only two operators — joins, filters, group-bys are
awkward
- High latency — overhead per job is large
- Java-heavy API — verbose for data exploraWon

### Hadoop
The open-source implementation that put MapReduce in everyone's hands.

MapReduce / Hive / Pig / Spark
- Processing engines

YARN
- Resource manager & scheduler

HDFS
- Distributed file system

Three layers: storage, resource management, computation.

### HDFS — Hadoop Distributed File System
Store a single huge file across many machines, with replication for safety

NameNode
metadata + directory
DataNode 1
blocks + replicas
DataNode 2
blocks + replicas
DataNode 3
blocks + replicas
DataNode 4
blocks + replicas
heartbeats + block reports
each block replicated 3× by default
NameNode knows what's where. DataNodes hold the bytes.

### How HDFS reads and writes a file
Files are split into blocks (typically 128 MB). Each block is replicated 3 times.

W R I T E
1. Client asks NameNode where to put block
#1
2. NameNode picks 3 DataNodes (locality-
aware)
3. Client streams block to first DataNode
4. First DataNode pipelines copies to the other
two
5. Acks flow back. Repeat for next block.

R E A D
1. Client asks NameNode for block locaPons
2. NameNode returns list of DataNodes per
block
3. Client reads each block directly from the
nearest replica
4. If a DataNode fails, retry from a replica
5. NameNode is not in the data path — only
metadata

Designed for huge sequen0al reads — not for many small files or random updates.

### What HDFS means for you as a data scientist
The file system shapes which file formats and access patterns actually work.

Prefer columnar formats
- Parquet / ORC store columns together — read only the columns you
need. Massive wins over CSV for analytics.

Aim for big files
- Many small files exhaust the NameNode. Aggregate into ~128 MB+
files. Coalesce before writing.

Append-only, immutable
- HDFS doesn't do random updates. Add new files for new data;
rewrite par^^ons for correc^ons.

Partition by query patterns
- Organize folders by date / region / etc. so jobs scan only relevant
partitions.

### YARN: who gets the cluster's CPUs?
Yet Another Resource Negotiator — the layer that schedules jobs on the cluster.

The problem
- Multiple jobs want to run on the same cluster — MR jobs,
Spark jobs, Hive queries. Who gets which machines? When?

YARN's role
- A central ResourceManager hands out containers (CPU + RAM
bundles) on worker NodeManagers. Each job gets its own
ApplicationMaster to coordinate its tasks.

Why it matters
- Decoupling resource management from execution let
MapReduce, Spark, Tez, Flink, and others share the same
cluster — the framework can change, the storage and
scheduling stay

### Spark
MapReduce's faster, friendlier successor.

MapReduce writes intermediate data to disk between every
job → Spark keeps intermediates in memory — orders of
magnitude faster for iteraTon

Itera8ve jobs chain dozens of MR jobs together → Spark expresses the whole computation as one
program with a DAG of steps

Only two operators (map and reduce) make joins ugly → Spark offers a rich API: filter, join, groupBy,
reduceByKey, ML, SQL

### Spark architecture in one picture
A driver program coordinates many executor processes on cluster workers.
Driver
Your program
Builds the DAG
Schedules tasks
Cluster
Manager
YARN / k8s / standalone
Executor 1
tasks • cached RDD parJJons
Executor 2
tasks • cached RDD parJJons
Executor 3
tasks • cached RDD partitionsrequests resources, then dispatches tasks
executors hold data in memory across tasks → fast iteration

### RDD: Resilient Distributed Dataset
Spark's core abstracMon — three leVers in the name, four key properties.

R
Resilient
- Auto-recovers from worker
failures using lineage

D
Distributed
- Partitioned across many
machines in the cluster

D
- Dataset
A collec8on of records —
rows, tuples, objects

+
Immutable
- Never modified;
transformations create new
RDDs

Mental model: an RDD is a logical handle to a partitioned, distributed collection. You describe
transformations on the handle — Spark figures out how to execute them.

### Transformations vs Actions
The single most important distinction in Spark.

T R A N S F O R M A T I O N S
Build a new RDD from another. Lazy.

map(f) — apply f to each element

filter(f) — keep elements where f is true

flatMap(f) — map then flaqen

groupByKey() — group values per key

reduceByKey(f) — combine per key

join(other) — relaGonal join on key

distinct() — unique elements


A C T I O N S
Trigger execution. Return a value or write output.

collect() — bring all elements to driver

count() — number of elements

first() — first element

take(n) — first n elements

reduce(f) — combine all elements

saveAsTextFile(p) — write to disk / HDFS

foreach(f) — apply f for side effect

Rule of thumb: transformations describe; actions execute.

### Lazy evaluation: Spark waits for an action
Transformations just build a plan. Nothing runs until you ask for a result

In [ ]:
# nothing executes here
lines = sc.textFile("hdfs://logs/*")
errors = lines.filter(lambda l: "ERROR" in l)
words = errors.flatMap(lambda l: l.split())
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a,b: a+b)
# this triggers everything above
counts.collect()

T h e D A G S p a r k b u i l d s
textFile
filter
flatMap
map
reduceByKey
collect
action ↓ triggers the chain

### Lineage: how RDDs recover from failure
Spark doesn't replicate data — it remembers how each parMMon was built.

The trick
- Every RDD remembers its parents and the transformation that created it.
This chain is the lineage graph.

When a worker dies
- Spark loses the cached partitions on that worker. It looks up the lineage,
finds where they came from, and recomputes only the missing pieces —
on another worker.

Why this is clever
- No replication overhead during normal execution. The cost is paid only on
failure, and only for the lost partitions.

Trade-off
- Long lineages get expensive to replay. Use checkpoint() to truncate the
graph for iterative jobs.

Recovering a lost partition
HDFS file
filter
map
reduceByKey
✗ par--on lost
recompute from upstream →

### A realistic Spark workflow
Loading logs, joining with metadata, computing a per-user metric

In [ ]:
# 1. Load — transformations, no work yet
events = sc.textFile("hdfs:/events/*").map(parse_json)
users = sc.textFile("hdfs:/users.csv").map(parse_user)
# 2. Shape into (user_id, value) pairs
ev_pairs = events.map(lambda e: (e.user_id, e.amount))
user_pairs = users.map(lambda u: (u.id, u.country))
# 3. Join and aggregate
joined = ev_pairs.join(user_pairs)
by_country= joined.map(lambda x: (x[1][1], x[1][0]))
totals = by_country.reduceByKey(lambda a,b: a+b)
# 4. Action triggers the whole plan
totals.saveAsTextFile("hdfs:/out/totals")

### When to reach for what
Different workloads call for different tools.

| Dimension  |  MapReduce | Spark  |
|---|---|---|
| Intermediate data  | Written to disk  | Kept in memory  |
|  IteraUve jobs | slow — many chained jobs  | Fast — one DAG  |
| API breadth  | map + reduce  | rich: filter, join, ML, SQL, streaming  |
|  Latency | Minutes per job overhead  | Seconds; sub-second with caching  |
| Best for  | huge one-shot batch over cold data  | Iteration, ML, interactive analytics  |
| Fault tolerance  | Re-run failed tasks  |  Lineage-based recomputation |

In practice today: most teams run Spark on YARN or Kubernetes, reading from HDFS
or cloud object stores.

### What to remember
01. BSP gave us the model
- Local work → communication → barrier. Everything else builds on this.

02. MapReduce made it usable
- Two func^ons plus a framework hide all the distribu^on work from the programmer.

03. Hadoop + HDFS made it free
- Open-source stack that scaled to thousands of nodes on cheap commodity hardware.

04. Spark made it fast
- In-memory RDDs, lazy DAGs, lineage-based recovery — a richer API on the same foundations.

N e x t : h a n d s - o n w i t h P y S p a r k o n a H a d o o p c l u s t e r .

## Lecture 12: Solving Problems with MapReduce

### Every MapReduce problem fits this template
Three questions to answer before you write a line of code.
1. How is the input split?
What is a “record”? How is the data sharded across workers? (Usually tuple-wise on HDFS blocks.)
2. What does Map emit?
For each record, what (key, value) pair(s)? The key controls who-talks-to-whom in the shuffle.
3. What does Reduce do?
For each unique key, what aggregation runs over its list of values? (Or no reduce — “Map-only”.)

Answer those three and the rest — shuffle, sort, fault tolerance — is the framework's job.

### The two costs that matter: disk and network

Disk I/O
- unit: pages read
- Each worker reads its shard from local disk.
- r u l e o f t h u m b
≈ total input size / page size
- e x a m p l e: 
6 pages of input on 3 workers = 6 disk-page reads total, 2 per
worker.

Network I/O
- unit: bytes shuffled
- Map outputs travel to reducers; partial results return to the
manager.
- r u l e o f t h u m b: 
depends on key cardinality + value size
- e x a m p l e: 
Simple aggregate sends 1 partial value per worker — tiny.
GROUP BY sends one per group.

We'll report both for every pattern: Disk: ___ pages Network: ___ pages

### SQL operations as MapReduce
Select • Project • Aggregates • GROUP BY

### Pattern: Relational SELECT
Filter tuples by a predicate — the simplest MR job, and the most common.

Input Split: Shard the table tuple-wise across worker disks.

map( ): For each tuple: if it satisfies the predicate, emit
(dummy_key, tuple).

reduce( ): Not needed — this is a Map-only job.

E x a m p l e q u e r y: 
SELECT * FROM R
WHERE A > 100

I / O c o s t: 
- Disk: N pages (one full scan)
- Network: 0

Why no shuffle? Each output tuple depends only on one input tuple — workers never need to talk. The output is
itself a sharded file, ready for the next stage.

### Example: filter rows where D > 5
Three workers, six pages of input, traced through Map.

 n p u t s h a r d s
Worker 1
(1, 4)
(2, 9)
Worker 2
(3, 7)
(4, 1)
Worker 3
(5, 8)
(6, 3)
L o c a l m a p ( )
predicate: D > 5
(2, 9) ✓
(1, 4) ✗
predicate: D > 5
(3, 7) ✓
(4, 1) ✗
predicate: D > 5
(5, 8) ✓
(6, 3) ✗
O u t p u t s h a r d s
part 1
(2, 9)
part 2
(3, 7)
part 3
(5, 8)
Disk: 6 pages • Network: 0 • Output: 3 pages (sharded, never re-assembled)

### Pattern: Non-deduplicating PROJECT
Keep only some columns — also a Map-only job.

Input Split: Shard the table tuple-wise across worker disks.

map( ): For each tuple: emit (dummy_key, projected_columns_only).

reduce( ): Not needed — Map-only. (Use DISTINCT? Then reduce will
have to output a set. One Reducer)

E x a m p l e q u e r y: 
SELECT B
FROM R

I / O c o s t:
- Disk: N pages (one full scan)
- Network: 0 

Why “non-deduplicating”? Plain Map can't dedupe — duplicates live on different workers. SQL's
DISTINCT requires a reduce step that groups by the projected row.

### Pattern: Simple aggregate
One global answer — SUM, MAX, AVG, COUNT.

- Input Split: Shard the table tuple-wise across worker disks.
- map( ): Compute partial stats over each shard. Emit (DUMMY,
partial_stats).
- reduce( ): One reducer receives all partial stats. Combine into the
final answer.

e x a m p l e q u e r y: 
SELECT MAX(A)
FROM R

I / O c o s t:
- Disk: N pages
- Network: # workers (tiny)

Combiner tip: use a single dummy key so all partial stats go to one reducer. Map already pre-aggregates per shard,
so network traffic is # workers — not # tuples.

### Example: SELECT MAX(A) FROM R
Each worker finds its local max, the reducer takes the max of those.

Worker 1 shard
A: [3, 7, 4, 1]
local map → 7
(_, 7)
Worker 2 shard
A: [2, 6, 9, 5]
local map → 9
(_, 9)
Worker 3 shard
A: [8, 1, 4, 6]
local map → 8
(_, 8)
reduce( ) : max(7, 9, 8) = 9
Disk: 12 tuples (full scan) • Network: 3 values (one per worker)

### Not all aggregates are equally parallel-friendly
Three categories — based on how much state Map needs to emit per shard.

Distributive
- s i z e e m i t t e d p e r s h a r d: 
1 value per shard
- e x a m p l e s: 
MIN, MAX, COUNT, SUM
- w h y: 
Same function works locally and
globally. Map emits the local
answer; Reduce applies the same
op again.

Algebraic
- s i z e e m i t t e d p e r s h a r d: 
O(1) values per shard
- e x a m p l e s: 
AVG, VARIANCE, STDEV
- w h y: 
Need a tuple of sufficient statistics.
e.g. AVG ships (sum, count);
Reduce divides at the end.

Holistic
- s i z e e m i t t e d p e r s h a r d: 
O(N) — no easy summary
- e x a m p l e s: 
MEDIAN, MODE, PERCENTILES
- w h y: 
No constant-size summary exists.
Workers may need to ship most of
their data — or use approximation.

MEDIAN(whole) ≠
f(MEDIAN(shard1),
MEDIAN(shard2), …)

### Pattern: GROUP BY aggregate
The grouping column becomes the Map output key.

Input Split: Shard tuple-wise as usual.

map( ): For each tuple: emit (grouping_column,
partial_stats_for_agg).

reduce( ): Per key, combine partial stats into final per-group
answer.

E x a m p l e q u e r y: 
SELECT A, SUM(D)
FROM R
GROUP BY A

I / O c o s t: 
- Disk: N pages
- Network: ~ #groups × #workers

Watch the cardinality. If A has millions of distinct values (high cardinality), the reducer hash table may not fit in
memory. Tune the number of reducers accordingly

### Worked example: SUM(D) grouped by A
Map emits (A, D) per row. Shuffle groups by A. Reduce sums each group.
I n p u t R
A D
a1 4
a2 3
a1 5
a3 1
a2 10
a1 8
M a p o u t p u t ( A , D )
(a1, 4)
(a2, 3)
(a1, 5)
(a3, 1)
(a2, 10)
(a1, 8)
S h u f f l e ( b y k e y )
a1 → [4, 5, 8]
a2 → [3, 10]
a3 → [1]
R e d u c e o u t p u t
a1 → 17
a2 → 13
a3 → 1
Reduce of group a1 is called once with [4,5,8]; a2 with [3,10]; a3 with [1]. Sums computed in parallel.

### What if the result doesn't fit in memory?
GROUP BY breaks down when the number of distinct groups is huge.

The problem: 
Reducer is building a hash table keyed by the grouping column. With high-
cardinality keys (URLs, user IDs, product SKUs), it can grow to billions of entries.

Three fixes
1. More reducers — partition the key space so each reducer holds only a slice
of the hash table.
2. Combiner — pre-aggregate within each Mapper so identical keys collapse
before the shuffle.
3. Spill to disk — modern frameworks page out the hash table to local disk if
RAM fills.

Skew warning: 
If one key has 90% of the rows (think: NULL, or a viral user), one reducer drags
the whole job

### Numeric and ML operations
Matrix sum • Lp norm • Batch Gradient Descent • Why SGD is different

### Pattern: Matrix sum / LP norm
Reduce a tiled matrix to a scalar — it's a simple algebraic aggregate.

- Input Split: Matrix tiled into blocks; each worker holds some tiles.
- map( ): Compute partial sum / partial Lp value over each tile.
Emit (DUMMY, partial).
- reduce( ): Sum all partials. For Lp norm: apply the final p-root at
the end.

M a t h r e m i n d e r – P N o r m:
‖M‖p = ( Σ |mij|^p )^(1/p)

the inner sum is associative — perfect for partial-sum MR

I / O c o s t:
- Disk: N pages
- Network: # workers (one scalar each)

Why this matters for ML: many loss functions and regularizers reduce to norms over a tiled parameter
matrix. Same MR pattern, different aggregator.

### Worked example: sum of a 6×4 matrix
Split into three 2×4 row-bands across 3 workers — each emits one partial sum.
Worker 1 • rows 1–2 • 8 cells
11 12 13 14
21 22 23 24
partial sum = 140
Worker 2 • rows 3–4 • 8 cells
31 32 33 34
41 42 43 44
partial sum = 300
Worker 3 • rows 5–6 • 8 cells
51 52 53 54
61 62 63 64
partial sum = 460
(_, 140) (_, 300) (_, 460)
reduce( ) : 140 + 300 + 460 = 900

### Pattern: Batch Gradient Descent
The gradient over a dataset is the sum of per-example gradients — another algebraic aggregate.

- Input Split: Shard the training set tuple-wise. Broadcast current
model θ to all workers.
- map( ): For each example: compute ∇L(θ; xᵢ). Sum within the
shard. Emit (DUMMY, partial_∇).
- reduce( ): Sum all partial gradients → full batch gradient. Update
θ ← θ − η·∇. Repeat.


M a t h r e m i n d e r: 
∇L(θ) = Σᵢ ∇ℓ(θ; xᵢ)

sum is associative → MR-friendly


P e r  e p o c h
- Disk: N pages
- Network: # workers × |θ|

One MR job per gradient step. Hadoop chains them as a workflow; Spark keeps θ cached and runs the loop in a
single application — much faster.

### Why SGD doesn't fit cleanly into MR
Mini-batches break the “sum is associative” trick that BGD relies on.

How SGD works
- Shuffle data, then process in tiny mini-batches.
- Update θ after each mini-batch.
- Many updates per epoch — hundreds or
thousands of θ revisions.
- Order of updates affects the trajectory (non-
commutative).
- Optionally re-shuffle every epoch.

Why MR can't handle it cleanly
- Sequential dependency: mini-batch k+1 needs θ
from mini-batch k. Can't parallelize across mini-
batches within an epoch.
- Not an algebraic aggregate: no constant-size
summary captures the work done in a shard.
- Order matters: different shard orderings give
different final θ — unlike SUM or MAX.

→ This is why ML systems invented new abstractions like the Parameter Server (next slide).

### Parameter Server: a beyond-MR pattern
Drop the barrier — accept some staleness — gain massive throughput on SGD.

The idea: 
Split the model θ across several Parameter Server (PS) nodes. Each holds a
slice.

Workers: 
Pull the latest θ slice they need, compute gradient on a mini-batch, push
update back.

Asynchrony: 
No barrier. Some workers see slightly stale θ — SGD turns out to be robust
to this.

Cost: 
Network traffic is high — workers and PS chat constantly.

### Advanced patterns
Chained jobs • K-Means • Joins • Inverted Index

### When one job isn't enough: chained MR
Many algorithms need multiple passes — wire them together as a workflow.

MR job 1
ETL: parse + clean
→
MR job 2
GROUP BY user
→
MR job 3
rank within group
→
Output
top-K per user
raw logs HDFS HDFS small final file

Hadoop pain: 
Every job writes its output to HDFS, the next reads it
back. That's 2× disk I/O per stage.

Spark fix: 
Stages share intermediate RDDs in memory. One
application, one DAG — no disk between stages

### K-Means as two MR jobs per iteration
Assignment step and Update step — each one a full MR pass.

Lloyd's algorithm: (1) assign each point to its nearest centroid; (2) recompute each centroid as the mean of its
assigned points. Repeat until convergence.

Job 1 — Assignment
- Read: centroid matrix A from HDFS (small, broadcast)
- map( ): compute dist to all k centroids; emit (point_id,
nearest_cluster_id)
- reduce: not needed — Map-only; output is new
assignment matrix B

Job 2 — Update
- Read: assignment matrix B from HDFS
- map( ): emit (cluster_id, point_vector)
- reduce: for each cluster: average all incoming vectors →
new centroid

Iterate jobs 1 → 2 → 1 → 2 … until centroids stop moving.

### K-Means: one iteration walked through
8 points, k=2. After one pass, centroids move toward their clusters.
d a t a p o i n t s + c e n t r o i d s
c1 old
c2 old
c1 new
c2 new

Job 1 map: p1 closer to c1_old → emit (p1, 1). p5 closer to c2_old →
emit (p5, 2). …

Job 1 out: new B: {p1→1, p2→1, p3→1, p4→1, p5→2, p6→2, p7→2,
p8→2}

Job 2 map: emit (1, p1), (1, p2), (1, p3), (1, p4), (2, p5), (2, p6), (2, p7), (2,
p8)

Job 2 reduce: avg of cluster 1 vectors → c1 new. avg of cluster 2 vectors →
c2 new.

### Joins in MR — Broadcast join (small × big)
When one table fits in memory, ship it everywhere and join locally.

- Setup: Broadcast small table S to every worker (e.g. via HDFS cache).
- map( ): For each tuple of big table B: look up join key in S's in-memory hash table;
emit joined row.
- reduce( ): Not needed — Map-only join!

E x a m p l e: 
events ⋈ users
users fits in RAM
(10K rows)

I / O c o s t
- Disk: |B| pages
- Network: |S| × #workers

Win: no shuffle of the big table. This is the fastest join when one side is small (config: spark.sql.autoBroadcastJoinThreshold).

### Joins in MR — Reduce-side join (big × big)
When neither table fits in memory, send both through the shuffle.

- Input Split: Shard both tables tuple-wise across workers.
- map( ): From A: emit (join_key, ('A', row)). From B: emit (join_key, ('B', row)).
- reduce( ): Per key: split incoming list by tag → A-rows × B-rows = joined output.

T r i c k: 
tag each row
with its source
table name

I / O c o s t
- Disk: |A| + |B|
- Network: |A| + |B| (full shuffle)

Expensive but general. Watch for skew on the join key — a popular value can overwhelm a single reducer. Mitigation: salt the key.

### Reduce-side join — a full worked example
Join Orders (A) with Customers (B) on name. Watch one key fan out.

① I N P U T
A · Orders
oid name
101 Alice
102 Bob
103 Alice
B · Customers
name country
Alice US
Bob UK
② M A P — t a g e a c h r o w b y s o u r c e
(Alice, ('A', 101))
(Bob, ('A', 102))
(Alice, ('A', 103))
(Alice, ('B', US))
(Bob, ('B', UK))
③ S H U F F L E — g r o u p b y k e y
key = Alice
('A', 101)
('A', 103)
('B', US)
key = Bob
('A', 102)
('B', UK)
④ R E D U C E — A × B p e r
k e y
name oid ctry
Alice 101 US
Alice 103 US
Bob 102 UK
Alice: 2×1 = 2 rows
Bob: 1×1 = 1 row
→ 3 joined rows

Cost on this example: Disk = |A|+|B| = 3+2 = 5 reads • Network = 5 tuples shuffled • every row crosses the network —
that's the price of big × big

### Classic example: building an inverted index
The original MapReduce use case at Google — search engine indexing.

- Input Split: Each record is one document (id, text).
- map( ): For each word in the document: emit (word, doc_id).
- reduce( ): Per word: collect all doc_ids into a posting list. Output (word, [doc_ids]).

m i n i e x a m p l e:

Input:
d1: "the cat sat"
d2: "the dog ran"
Map emits:
(the, d1) (cat, d1)
(sat, d1) (the, d2)
(dog, d2) (ran, d2)
Reduce output:
the → [d1, d2]
cat → [d1]
sat → [d1]
dog → [d2]
ran → [d2]

### Hybrid Parallelism
Mixing task parallelism with BSP data parallelism


### Task vs Data Parallelism — recap
Two paradigms, two cost profiles. Pick the wrong one and you waste resources.

Task parallelism
(e.g. Dask, multi-process)
P R O S
- Easy to implement
- No shuffles needed
- Different tasks fully independent

C O N S
- Replicates data on every node
- Wastes memory and storage
- Remote reads → network burn

BSP data parallelism
(e.g. MapReduce, Spark)

P R O S
- Scales without data replication
- Cheap per-machine memory
- Handles huge datasets

C O N S
- Painful to implement per op
- Network costs for shuffle
- Barrier means stragglers stall everyone

Q: Can we have the best of both?

### A worked scheduling problem
Four independent tasks share one dataset across four nodes.

Four tasks (T1–T4) operate on the same dataset D

A s s u m p t i o n s
- 4 worker nodes available
- Each task gets perfect linear speedup if data-parallel
- Manager overhead = 1 unit before, 1 after each data-par
step
- All tasks read the same dataset D

Q: What's the shortest possible total time?

### Schedule 1 — Fully task-parallel
One task per node, run end-to-end. Best case: each task on its own node.

Total: ~ 30 units. Half the cluster is idle; one node carries the whole long task.

### Schedule 2 — Fully data-parallel (BSP)
All 4 nodes work on T1, then all on T2, etc. Total: 20 units.

Total: 20 units. All workers used, but tasks run sequentially with barriers between.

### Schedule 3 — Hybrid: best of both
Run the cheap tasks task-parallel while the big one runs data-parallel. Total: 18 units.

-par on W1+W2 || T3 task-par on W3 → T4 data-par
Total: 18 units. Task-par 30 • Data-par 20 • Hybrid 18 → smallest wins

### Hybrid parallelism in practice
Software complexity is high, so few systems support it well — but the gains are real.

Where it stands today
- Most systems pick one paradigm.
- Dask → task-parallel only. Hadoop / RDBMS → data-parallel
only.
- Multi-query execution.
- Some RDBMSs run different queries task-parallel over shared
sharded data — a form of hybrid for SQL workloads
- Spark moving toward hybrid.
- Recent Spark versions support some task-parallel execution
alongside data-parallel stages.

Research spotlight: Cerebro

Deep-learning model selection on clusters (UCSD, 2021).

The challenge: 
Training many DL configurations in parallel — each one is itself a data-parallel
job.

The hybrid solution: 
Different model configs run task-parallel across nodes. Within each config,
data-parallel over a shard. Workers swap shards (not gradients) between
epochs.

The result: 
First known form of “Bulk Asynchronous Parallelism”. Resource-optimal
across compute, memory, and network

### S U M M A R Y • W H A T T O R E M E M B E R
Solving problems with MapReduce

01. Three questions: 
Input split, Map key/value, Reduce aggregation — always answer these first.
02. Patterns repeat: 
Select, project, aggregate, GROUP BY, matrix sum, gradient — same template, different aggregator.
03. Costs are I/O: 
Disk pages for the scan, network bytes for the shuffle. Compute is rarely the limit.
04. MR has limits: 
Holistic aggregates, SGD, big-cardinality GROUP BY — need approximation, async, or new patterns.
05. Hybrid wins sometimes: 
Mixing task and data parallelism can beat either alone — when scheduling complexity is worth it.

N e x t : h a n d s - o n w i t h P y S p a r k — i m p l e m e n t t h e s e p a t t e r n s y o u r s e l v e s .

## Lecture 13: MLOps Systems

### Model Building

Model building is a four-stage pipeline


01. Feature
Engineering: 
Turn raw data into informative
inputs the model can use.

02. Algorithm
Selection: 
Choose a model family whose
assumptions fit the problem.

03. Hyperparameter
Tuning: 
Set the dials that control how
the model learns.

04. Testing:
Estimate behavior on data the
model has never seen.

It is a loop, not a line. Test results send you back to re-engineer features or re-tune. The arrows point
right, but you walk them many times.

### A feature is a measurable input the model learns from

The core idea
- Raw data is rarely in a form a model can use well.
- Feature engineering reshapes raw fields into numeric
inputs that expose the signal.
- Good features make the relationship the model must
learn simpler and more linear.
- It encodes domain knowledge the algorithm cannot
discover on its own.

From raw to useful
|  Raw field | Engineered feature  | Why  |
|---|---|---|
| timestamp 2024-03-09 18:42   | hour=18, is_weekend=1   | captures daily/weekly rhythm  |
| address "San Diego, CA"  | latitude, longitude, density  | turns text into geometry  |
|  price in dollars | log(price)  | tames a skewed long tail  |
| height, weight  | BMI = wt / ht²   | domain ratio the model needn't relearn  |

### Feature work is the first layer of a task graph

Think in tasks, not steps
- A task is one unit of work with a runtime.
- Each feature-engineering (FE) approach is a
task.
- Independent tasks can run on different
workers at the same time — task
parallelism.
- A task can only start once the tasks it
depends on have finished.
- Build features once, reuse them across
every model that follows.

### More features is not always better

Filter methods: 
Rank features by a quick statistic (correlation,
mutual information) and keep the top ones.
Fast, model-agnostic.

Wrapper methods: 
Repeatedly train the model adding/removing
features (e.g. recursive elimination). Accurate
but expensive.

Embedded methods: 
Selection happens during training — L1/Lasso
zeroes out weak weights; trees report feature
importance.

Why prune at all?
- Fewer features → faster training and prediction, essential at scale.
- Removes noise columns that let the model memorize rather than generalize.
- Simpler models are easier to explain, audit, and maintain.

### Algorithm Selection - Every choice trades bias against variance

High bias (underfit)
- Model too simple to capture the
pattern.
- Misses signal in both training and
test data.
- Symptom: poor on training set
too.
- Fix: richer model, more features.

The sweet spot
- Captures real structure, ignores
noise.
- Training and test scores are close
and good.
- Found by tuning complexity, not
luck.
- This is the goal of every later
stage.

High variance (overfit)
- Model memorizes training noise.
- Great on training, poor on new
data.
- Symptom: big train–test gap.
- Fix: regularize, simplify, more
data.

### T H E G U I D I N G P R I N C I P L E
Averaged over all possible problems, no algorithm
beats any other. A method only wins because its built-
in assumptions happen to match the structure of your
specific data.

So what do we do?
Start simple, establish a baseline, then add
complexity only if it pays off in testing.
Match assumptions: 
Linear for linear, trees for tabular
interactions, nets for raw signals.

Empirically compare: 
Shortlist 2–3 families and let cross-validated
results decide.

### Each family carries a different compute cost

| Family  | Training cost  | Parallelizes?  |
|---|---|---|
| Linear / logistic  | Cheap — fast even on large data   | Across data partitions  |
| Tree ensemble (RF)  | Moderate — each tree is independent  | Easily — trees built in parallel  |
| Boosting  | Moderate–high — trees built in sequence   | Within a tree, not across trees  |
| SVM / kNN  |  Grows fast with rows (can be quadratic) | Poorly at large scale  |
| Neural net  | Expensive — needs GPUs  | Across data and devices  |

Why it matters here
- Random Forest = many independent tree-builds
→ embarrassingly parallel.
- Naive Bayes is a single cheap pass over the
data.
- These costs become the task runtimes you
schedule when you tune

### Five questions before you pick

1. What is the data type? Pixels/audio/text → neural nets. Rows of mixed columns → tree ensembles or linear.
2. How much data do you have? Small → linear/SVM. Large & unstructured → deep learning.
3. Do you need to explain it? Regulated or high-stakes decisions favor linear models or simple trees.
4. What are the compute limits? Training time, memory, and latency at serve time constrain the realistic options.
5. Is the relationship linear? If yes, a linear model may win outright; if not, lean to trees or nets

### Hyperparameter Tuning
You've picked a family and built features. Hyperparameters are the settings you
choose before training — getting them right can matter as much as the
algorithm itself.

The dials that control how a model learns

### Parameters are learned; hyperparameters are chosen
Parameters
- Learned automatically from the data during training.
- The weights in a linear model; the split points in a tree.
- You never set these by hand.
- There can be millions of them.

Hyperparameters
- Set by you BEFORE training begins.
- Control model capacity and how learning proceeds.
- Tree depth, learning rate, regularization strength, number of
neighbors.
- Tuning = searching for the best combination.

### What you actually tune, by problem

| Problem class  | Highest-impact dials   | Validation scheme  |
|---|---|---|
| Tabular regression  | Tree depth, learning rate, # trees, L2   | Standard k-fold  |
|  Imbalanced classification | Class weight, decision threshold, depth  | Stratified k-fold (preserve rare class)  |
| Image classification  | Learning rate, augmentation strength, dropout   |  Hold-out + early stopping |
| Time-series forecasting   | Lag window length, learning rate, regularization  | Forward-chaining (time-ordered) splits  |

Notice: the validation scheme changes with the problem. Imbalanced data needs stratified folds; time-series must
never shuffle.

### Testing & Evaluation
Training accuracy means nothing. The real question: how will this model behave
on data it has never seen? Choosing the right metric is half the battle.

A model is only as good as its honest evaluation

### The right metric depends on the problem

| Problem class  | Primary metric  | Why this one  |
|---|---|---|
|  Tabular regression (price) | RMSE / MAE / R²  | Continuous error in interpretable units  |
|  Imbalanced classification (fraud) | PR-AUC, recall @ fixed precision   | Accuracy & ROC mislead on rare positives  |
| Image classification (scans)  | Recall + confusion matrix by class  | Missing an abnormality (FN) is the costly error  |
| Time-series (electricity)   | RMSE on a forward hold-out, MAPE   | Must score on genuinely future, unseen periods  |

The single biggest testing mistake: optimizing accuracy on an imbalanced problem. Always ask what error actually
costs before choosing a metric

### Systems & MLOps
for Model Building & Deployment

### Model building is a system, not a notebook
In a notebook, the only question is: does the model work? In production, three more questions: how fast, how big, how
much?

Accuracy: 
Does it do the task well — on data it
has never seen?
The traditional ML question.

Performance: 
How fast does training finish; how
quickly does each prediction return?
Set by parallelism, hardware, and the
critical path.

Cost: 
What does all of this cost — in
compute, storage, and engineer-
hours?
Linear in workers; spikes with scale

### Training and serving are opposite-shaped systems
TRAINING
- Goal Fit parameters from data
- Runs Occasionally (hours–days)
- Pattern Bursty, batch-heavy
- Key metric Throughput — get done fast
- Hardware Many cores / GPUs
- Cost shape Big spikes, then idle

SERVING
- Goal Answer requests with the trained model
- Runs Continuously, 24×7
- Pattern Steady stream of small requests
- Key metric Latency — answer fast each time
- Hardware Right-sized replicas, always on
- Cost shape Flat baseline, scales with traffic

Same model, two completely different engineering problems.

### P A R T I The Training Workload
Training is a one-off (or periodic) batch job: many
independent and dependent tasks, finishing as fast
as your workers and your critical path allow

### A training workload is a directed graph of tasks
What counts as a task?
- One unit of work with a runtime: a script,
a fit() call, a batch.
- A task has inputs (data, prior tasks' outputs)
and one output artifact.
- Tasks form a DAG: an edge means 'must
finish before'.
- Independent tasks can run on different
workers at the same time.
- Independent ≠ free: each one still costs
CPU/GPU time. => Parallelizable

### What sets the fastest possible finish time
Critical path: 
The longest chain of tasks that must run one after another,
dictated by the dependency edges in the graph.

= lower bound on time

 Even with infinite workers, you
cannot finish sooner.


Maximum concurrency:

The most tasks that are simultaneously ready to run — the
widest level of the graph.

= w orkers w orth having 

Adding workers beyond this width
buys nothing.

Two rules every workload obeys
1. Fastest finish time ≥ critical-path length.
2. Workers needed = maximum concurrency. Beyond that they sit idle.

To go faster than the critical path you must change the graph — not add machines.

### Counting workers and finding the critical path

Setup
- 4 feature-engineering (FE) approaches.
- Two Model Types
    - a Random Forest with 3 hyperparameter settings,
and
    - a Naive Bayes model.
- Runtimes:
    - FE = 25, 30, 40, 55
    - RF builds = 20, 35, 45
    - NB = 40

Q1 — workers needed (any runtimes)

After the FE layer, each of 4 FE sets feeds 4 models. All 4 ×
4 = 16 model builds are independent.

=> 16 workers
= maximum concurrency (widest level)

Q2 — upper bound on completion time

With enough workers, each branch finishes at FE-time +
slowest model build. Slowest branch: 55 + 45.

100 units
= critical-path lengt

### Throughput and latency at training time
Throughput: = w o r k / t i m e

Work done per unit time — examples/second,
epochs/hour, models trained per day.

Maximized by large batches, parallel workers, full GPUs.

Latency: = t i m e p e r j o b

Time from job start to job finish — the wall-clock of one
training run.

Minimized by smaller per-task work, more workers on the
critical path.

They trade against each other
- Bigger batches → higher throughput per epoch, but each step takes longer (worse step latency).
- Splitting one model across more workers → lower latency per run, but coordination overhead drops throughput.

A research team training many models cares about throughput. A team waiting on a single
result cares about latency.

### Three regimes — each with its own cost knee

| Regime  |  Best when | Throughput  | Coordination cost  | Cost shape  |
|---|---|---|---|---|
| Single CPU / GPU  | Small models, prototyping  |  Low | None  | Cheap, slow  |
| Multi-GPU, one node  | Large models that fit on one box  | High  | GPU-to-GPU on PCIe / NVLink   |  Big spike, short |
| Multi-node cluster  | Very large models or huge data  | Highest  | Network is the bottleneck  | Expensive, complex  |
|  Spot / preemptible | Tolerant, restartable jobs  | Same as base  | Must checkpoint frequently  |  60–90% cheaper |

The knee: each step-up doubles complexity but usually less than doubles speedup. Stay one tier below what you think you need.
Coordination cost — the time workers spend talking instead of computing — is what kills naive scale-out.

### Two ways to split a single training run
Data parallel

Every worker holds a full copy of the model. They process
different shards of each batch, then sync gradients.
- Most common scheme.
- Speedup ≈ N until sync time dominates.
- Memory per worker = the model size.

Model parallel
The model itself is sliced across workers. Each holds a
different layer or shard of parameters.
- Needed when the model is too large for one device.
- Communication on every forward/backward pass.
- Used by very large LLMs and giant CNNs.

Practical sequence: single-GPU → multi-GPU data-parallel → multi-node data-parallel → model-parallel only when you must.

### How jobs queue, and why your real wall-clock is slower
FIFO: 
First job in runs first; later jobs wait
their turn.

Simple, but one big job blocks
everyone behind it.

Fair-share:
Each user/team gets a slice of cluster
capacity; the scheduler interleaves
work.

Better latency for most; throughput
stays high.

Gang scheduling:
A multi-worker job runs only when
all its workers are free at once.

Required for distributed training —
avoids partial starts.

Your real wait time is more than your job's runtime

wall_clock = queue_wait + compute + startup + data_load

In a busy cluster, queue_wait can dwarf compute. Right-sizing your request matters as much as right-sizing the model.

### Compute time × hourly rate, and the spot-instance trade
$ = workers × hours per worker × $/hour for that instance type

Worked: training a deep model
- Workers 8 GPUs (1 node)
- Runtime 12 hours wall-clock
- Rate $3.06 / GPU-hour
- Subtotal 8 × 12 × $3.06
- Cost $ 293.76

Multiply by the number of runs you make per week, then by 52, to
see the annual training spend.

Spot vs on-demand
- On-demand: pay the full posted rate; cloud cannot evict you.
- Spot / preemptible: 60-90% cheaper; cloud may reclaim with
minutes' notice.
- Net cost = base_rate × (1 − discount) × restart_overhead.
- Worth it when: job is checkpointable, idempotent, and not on
the critical path.
- Avoid when: one long un-checkpointed run, or hard deadline.

### The Serving Workload
Inference is a different system: instead of finishing one big job fast, you must
answer many small jobs fast, forever, at a predictable cost per request.

### Online inference and batch inference — different SLAs entirely
Online inference: 
One request comes in → one prediction goes out,
immediately.

- Latency target tens of ms
- Pattern Many small requests, 24×7
- Driven by User-facing apps, transactions
- Scaling Add replicas as QPS grows

Batch inference: 
A pile of inputs is scored together, once a day or once an
hour.

- Latency target minutes to hours
- Pattern Scheduled jobs, big bursts
- Driven by Reports, downstream ETL
- Scaling Larger cluster for the run

Pick the pattern that matches your traffic shape. The wrong choice triples your bill or breaks your SLA.

### Why averages lie — p50, p95, p99, and the tail
Percentile latency
- p50 (median): half of requests are this fast or
faster.
- p95: 95% of requests are this fast or faster.
- p99 / p99.9: the slowest 1 in 100 / 1 in 1,000
requests.
- The tail (p99+) is usually 5–20× the median —
and it's what users notice.
- Garbage collection, cache misses, network blips,
cold replicas all live in the tail.
- SLA is set on a percentile, not an average.

###  H R O U G H P U T
Batching: 
Group N incoming requests into a
single call to the model. One GPU
pass scores all of them.

Big throughput win — but waits for
the batch to fill, adding latency.

L a t e n c y ↑ T h r o u g h p u t ↑ ↑

Concurrency: 
Run multiple requests in flight on
one replica (threads, async,
batched). Use CPU while GPU runs.

Free throughput when the model has
idle stages.

L a t e n c y ≈ T h r o u g h p u t ↑


Replicas: 
Run K identical copies behind a load
balancer. Requests fan out across
them.

Linear cost; doesn't help if a single
request is already too slow.

L a t e n c y = T h r o u g h p u t ↑ × K


Order matters: tune batching and concurrency first — they're free. Add replicas only when you've maxed each one out.

### How many replicas to hit 1,000 requests / second? 19
Given:
- Target load: 1,000 requests / second (QPS).
- One replica's p95 latency at light load: 50 ms.
- At full load each replica handles ≈ 60 RPS before queueing
hurts the tail.
- SLA: p95 latency ≤ 100 ms, with 30% headroom for
traffic spikes.

Reasoning
- Base need = 1,000 / 60 = 17 replicas.
- + 30% headroom → 17 × 1.30 ≈ 22 replicas.

Answer
22 replicas
to meet the SLA with headroom

Gotchas to budget for
- Cold-start time when scaling up.
- Variability in request size (longer inputs cost more).
- Failure of any single replica must still leave you above
target.

Sizing is arithmetic: RPS ÷ per-replica capacity × headroom — then verify with a load test.

### Hardware fit by model class — through the cost-per-request lens

| Model class  | Best hardware  |  Cost per request | Why  |
|---|---|---|---|
|  Linear / tree models |  CPU | Lowest  | Tiny ops, no benefit from a GPU  |
| Small neural net (< 100M params)  | CPU or small GPU  | Low  |  GPU only helps if you can batch heavily |
| Large neural net (CNN, BERT-sized)   | GPU  | Medium  | hroughput needs SIMD-style parallelism  |
| LLM via local serving  | Multi-GPU node  | High  | KV-cache + tensor parallel; needs the memory  |
|  LLM via vendor API | Someone else's  | Per-token rate  | You pay for tokens; they own the metal  |
| Edge (phone, browser)  | On-device  |  ≈ zero cloud cost | Latency + privacy; constrained memory  |

Cost per request, not per machine. A $30k/month GPU node that serves 5,000 QPS costs less per call than a
$300/month CPU that serves 5

### Autoscaling — when it pays, when it bites
Where autoscaling shines
- Diurnal traffic (busy daytime, quiet at night).
- CPU-served small models with sub-second startup.
- Batch inference: spin up a big cluster, run, shut it
down.
- Cost savings of 40-70% on cyclic loads are
common.

Where it bites
- Cold starts: loading a GPU model takes 30 s — 5
min.
- Spiky traffic outruns the scale-up rate; your p99
explodes.
- Scale-to-zero saves money but turns the first
request after idle into the worst latency of the
day.
- Mitigation: warm pool of replicas + predictive
scaling.

The default safe setting: scale on rolling p95 latency, not just CPU — that's what actually breaks the SLA.

### Caching, precomputation, and the feature store at serve time
Result cache: 
If the same (or similar) input comes in often, store the
prediction and skip the model.

Huge wins on hot keys: top products, top users, repeat
queries.

Precompute offline: 
If the universe of inputs is small enough, score
everything in a nightly batch and look up.

Recommender systems for catalogs. Risk: staleness.

Feature store:
Features computed in batch are ready at serve time —
the model only does the final fit step.

Cuts request latency and guarantees train/serve
consistency.

Quantize / distill: 
Smaller, cheaper model trained to mimic the big one.
Same answers, fraction of the cost.

Common: 8-bit weights, INT4 LLMs, MobileNets at the
edge.

### The Data Plane
Before the model can train or serve, the right rows have to land in the right
place at the right time. Feature pipelines are the system underneath the
system.

### Batch and streaming features — different latency, different cost

Batch features

Computed on a schedule (nightly, hourly) over a snapshot
of the warehouse.

- Freshness Hours to a day old
- Latency Cheap reads at serve time
- Cost shape Periodic batch job
- Examples Lifetime spend, 30-day rolling avg

Streaming features

Computed as events arrive — windowed aggregates
updated in real time.

- Freshness Seconds to minutes
- Latency More complex serve-time path
- Cost shape Always-on stream processor
- Examples Last-5-min click count, current cart

Most real systems are hybrid: stream the few features that must be fresh, batch the rest. Cost follows freshness.

### Train/serve skew — the most expensive silent bug
Skew = the same feature is computed differently in training and serving. The model learns one thing and gets another at run time. Accuracy
quietly collapses.

In training
- Features built in a SQL job over the warehouse.
- Joined with labels in pandas / Spark.
- Result: one CSV / parquet → train()
- avg_order_value computed from the orders table.

At serve time
- Features built by a Python service from API calls.
- Rounding, time-zones, null handling — subtly
different.
- Result: a feature vector built per request
- avg_order_value computed from cached recent
orders only.

Mitigations: shared feature store (one definition, two readers) · log live features and replay them in training · serve-time
monitoring that flags drift between expected and actual feature distributions.

### Orchestrators run the DAG of jobs that produce your
model
26
A typical training pipeline (Airflow / Kubeflow shape)
Ingest
Validate
Features
Train
Evaluate
Register
Canary
Promote
D a t a p l a n e T r a i n i n g
R e l e a s e

What the orchestrator does
- Schedules each task and tracks the DAG.
- Retries failed jobs, skips already-done
ones.
- Logs runtimes, artifacts, and lineage.
- Triggers downstream jobs on success.
- Enforces resource quotas and queues.
- Same tooling: Airflow, Kubeflow, Argo,
Prefect

### MLOps & Operations

A live model is software that decays. Closing the loop — from data through
deployment and back to retraining — is what keeps it useful and affordable
over time.

Deploying a model is the start, not the end

### Data → Train → Register → Deploy → Monitor → Retrain

Data
Train
Register
Deploy
Monitor
Retrain
MLOps
loop

What keeps the loop closed
- Automation: every step is a script,
not a person.
- Lineage: artifacts and metrics tied to
code & data versions.
- Triggers: monitoring fires retraining;
no manual ticket.
- Rollback: promotion is reversible if
metrics dip.

### Code + Data + Environment + Config = a reproducible run
Code: 
Versioned in git. A commit SHA pins
the exact transformation that
produced the model.

Data: 
Pinned to a snapshot ID, partition
date, or content hash. 'The same
query last Tuesday' is not pinning.

Environment: 
Container image with Python
version, library pins, and CUDA. 'It
worked on my laptop' is not a
deployment plan.

Config: 
Hyperparameters, seeds, paths.
Recorded with the run, not buried
in notebooks.

If you cannot rerun the model bit-for-bit, you cannot debug it, audit it, or trust it in production.
Reproducibility is a system property, not a developer virtue.

### CI/CD for models — like code, but with weights
Model registry
- A versioned store of trained models, like a
package registry for binaries.
- Each version carries metrics, lineage (data + code
SHA), and a stage tag.
- Stages: staging → canary → production →
archived.
- One source of truth for 'what is live right now'.
- Examples: MLflow, SageMaker Model Registry,
Vertex AI Model Registry.

Promotion flow
1. Train: 
Produce v1.4 from code SHA abc123 + data 2026-05-15
2. Validate: 
Offline metrics vs holdout — must beat current prod
3. Canary: 
Route 5% of traffic to v1.4 alongside v1.3
4. Monitor: 
Watch p95 latency, error rate, model metrics for N hours
5. Promote: 
If green, route 100%. If not, roll back instantly.

### Four things to monitor — only one is 'is the model right'
Input drift: 
The feature distributions shift over time —
users, products, seasons change.
Train data no longer looks like serve data → quiet
accuracy loss.

Prediction drift: 
The distribution of model outputs shifts —
more positives than usual, scores skewed.
An early warning, since input drift hits predictions
before it hits labels.

Model quality: 
Live accuracy / precision / recall — measurable
only once you get the ground truth.
Often delayed days or weeks; you cannot wait for it to
act.

System metrics: 
Latency p95/p99, throughput, error rate,
CPU/GPU, queue depth, cost per request.
Same metrics any web service watches — but now they
imply ML-specific causes.

Drift signals arrive before quality signals — that's why both layers must be watched

### Retraining triggers — three patterns and their costs

| Trigger  | How it works  | Pros  | Cons / cost  |
|---|---|---|---|
| Scheduled  | Retrain on a fixed cadence (nightly, weekly)  | Predictable; simple to operate  | Wasted compute if nothing changed; lags fast drift  |
| Drift-triggered   | Retrain when input/prediction drift exceeds a threshold  | Responsive; spends only when needed  | Threshold tuning is itself an ML problem  |
| Performance-triggered  | Retrain when live quality drops below an SLO  | Tied directly to business outcome  | Reactive — waits for damage; needs labels  |
| Hybrid (typical)  | Schedule + drift floor + emergency override  | Best of all three  | Most pipelines to maintain  |

Retraining is the most expensive recurring item in ML budgets. Pick the cheapest trigger that meets your quality bar — then add
safeguards above it.

### Total cost of ownership over a year, not just the GPU bill

Illustrative annual TCO for one production model
Serving compute 35%
Training compute 12%
Data + features 15%
Engineer time 25%
Monitoring + ops 8%
Storage & misc 5%
Compute is rarely the biggest line. Engineer-time is.

Levers that actually move
the bill
- Right-size the serving tier (not over-
provisioned).
- Use spot for restartable training.
- Cache hot predictions; precompute
when possible.
- Retire models that no team uses.
- Quantize / distill heavy models.
- Cut retraining cadence to match real
drift.

### Three model shapes, same systems axes

| Axis | Small classical model | Large in-house neural net | LLM via vendor API |
|--|--|--|--|
| Training cost | Minutes on CPU; cents | Hours on multi-GPU; thousands of $ | None (or fine-tune fee) |
| Serving hardware | CPU replica fleet | GPU replicas, often dedicated nodes | Vendor's; you pay per token |
| Latency profile | p99 in single ms | p99 tens of ms with batching | p99 hundreds of ms — variable |
| Scaling lever | Add CPU replicas | Add GPU replicas; quantize | Buy more rate-limit headroom |
| Cost shape | Flat, very cheap | High baseline + traffic | Linear in tokens |
| MLOps burden | Standard pipeline | Heavy: data, training, serving, monitoring | Light infra; heavy prompt + eval ops |
| When to choose | Most tabular problems | Unstructured data, owned model required | Fast start, no in-house ML team |


The choice is rarely about accuracy alone. It's about latency, hardware, ops burden, and the cost curve you can live with — every
column above is a systems decision.

### Six things to carry forward
Treat ML as a system. Training and serving are different workloads — design them as such.

Count the graph. Critical path bounds time; max concurrency bounds workers worth having.

Latency and throughput trade. You optimize one by giving up the other. Pick the metric your users feel.

Match the metal. Hardware fit drives cost per request more than raw model speed does.

Close the loop. Monitoring, retraining, and rollback are not extras — they're the product.

Engineer-time is the largest line. Compute is visible; people-cost is not. Optimize for ops simplicity.

A live model is software that decays. The systems around it are what make it useful, affordable, and trustworthy over time